### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="lung_cancer_epithelial_genexp",
    dataset_year="2006",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="GOV Website",
    original_dataset_source_download_link="https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE4115",
    download_description=r"""
We download the original gene expression data from the GEO records.

mkdir -p local-data-warehouse/lung_cancer_epithelial_genexp/ && wget -P local-data-warehouse/lung_cancer_epithelial_genexp/ https://ftp.ncbi.nlm.nih.gov/geo/series/GSE4nnn/GSE4115/matrix/GSE4115_series_matrix.txt.gz && gzip -d local-data-warehouse/lung_cancer_epithelial_genexp/GSE4115_series_matrix.txt.gz
""",
    # References
    academic_reference_bibtex=r"""@article{spira2007airway,
  title={Airway epithelial gene expression in the diagnostic evaluation of smokers with suspect lung cancer},
  author={Spira, Avrum and Beane, Jennifer E and Shah, Vishal and Steiling, Katrina and Liu, Gang and Schembri, Frank and Gilman, Sean and Dumas, Yves-Martine and Calner, Paul and Sebastiani, Paola and others},
  journal={Nature medicine},
  volume={13},
  number={3},
  pages={361--366},
  year={2007},
  publisher={Nature Publishing Group US New York}
}
""",
    academic_reference_bibtex_key="spira2007airway",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We start with an extended dataset of 192 patients. The original reference uses 129 samples as "Individuals without final diagnoses as of May 2005 were excluded from this primary dataset". We remove 5 samples without a clear label ("Smoker with suspect lung cancer Sample"), yielding a dataset of 187 samples.
- We use "Sample_title" from the metadata as labels for the prediction task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DiagnosedCancer",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="DiagnosedCancer",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "GSE4115_series_matrix.txt", sep="\t", comment="!", index_col=0).T
# read labels
with open(dataset_mold.path / "GSE4115_series_matrix.txt") as f:
    for line in f:
        if line.startswith("!Sample_title"):
            labels = [t.strip('"') for t in line.strip().split("\t")[1:]]
            break

df["DiagnosedCancer"] = pd.Series(labels, index=df.index)
def resolve_label(label):
    if label.startswith("Smoker diagnosed with cancer Sample"):
        return "Yes"
    if label.startswith("Smoker NOT diagnosed with cancer Sample"):
        return "No"
    if label.startswith("Smoker with suspect lung cancer Sample"):
        return np.nan
    raise ValueError(f"Unknown label: {label}")

df["DiagnosedCancer"] = df["DiagnosedCancer"].apply(resolve_label)
df = df.dropna(subset=["DiagnosedCancer"])  # remove suspected cancer samples
df["DiagnosedCancer"] = df["DiagnosedCancer"].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary.head(100)

In [ ]:
# Numeric Feature Statistics
numeric_stats.head(100)

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()